In [ ]:
import pandas as pd
import gradio as gr

In [ ]:
data = pd.read_csv("/content/sentiment_analysis.csv")

In [ ]:
data

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram
...,...,...,...,...,...,...,...
494,2015,10,18,night,"According to , a quarter of families under six...",negative,Twitter
495,2021,2,25,morning,the plan to not spend money is not going well,negative,Instagram
496,2022,5,30,noon,uploading all my bamboozle pictures of facebook,neutral,Facebook
497,2018,8,10,night,congratulations ! you guys finish a month ear...,positive,Twitter


In [ ]:
data.isnull().sum()

,0
Year,0
Month,0
Day,0
Time of Tweet,0
text,0
sentiment,0
Platform,0


In [ ]:
data.duplicated().sum()

np.int64(105)

In [ ]:
data = data.drop_duplicates()

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 394 entries, 0 to 498
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           394 non-null    int64 
 1   Month          394 non-null    int64 
 2   Day            394 non-null    int64 
 3   Time of Tweet  394 non-null    object
 4   text           394 non-null    object
 5   sentiment      394 non-null    object
 6   Platform       394 non-null    object
dtypes: int64(3), object(4)
memory usage: 24.6+ KB


In [ ]:
data = data.drop(columns=['Year', 'Month', 'Day','Time of Tweet','Platform'])

In [ ]:
data

,text,sentiment
0,What a great day!!! Looks like dream.,positive
1,"I feel sorry, I miss you here in the sea beach",positive
2,Don't angry me,negative
3,We attend in the class just for listening teac...,negative
4,"Those who want to go, let them go",negative
...,...,...
494,"According to , a quarter of families under six...",negative
495,the plan to not spend money is not going well,negative
496,uploading all my bamboozle pictures of facebook,neutral
497,congratulations ! you guys finish a month ear...,positive


In [ ]:
import string
import nltk
import re
from nltk.corpus import stopwords
nltk.download('stopwords')
def Text_Preprocessor(text):
  text = text.lower()
  text = re.sub("\d","",text)
  text = re.sub("\s+"," ",text)
  text = text.encode("ascii","ignore").decode("ascii")
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = " ".join([a for a in text.split() if a not in stopwords.words('english')])
  return text

<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_4118/3988448048.py:8: SyntaxWarning: invalid escape sequence '\d'
  text = re.sub("\d","",text)
/tmp/ipykernel_4118/3988448048.py:9: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("\s+"," ",text)
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
data['text'] = data['text'].apply(Text_Preprocessor)

In [ ]:
print(data.text[0])

great day looks like dream


## Tokenization

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
tok = Tokenizer()

In [ ]:
tok.fit_on_texts(data["text"])

In [ ]:
data["text"] = tok.texts_to_sequences(data["text"])

In [ ]:
data.sentiment.value_counts()

,count
sentiment,
neutral,153
positive,132
negative,109


In [ ]:
sentiment_map = {
    'negative': 0,
    'positive': 1,
    'neutral': 2,
}
data['sentiment'] = data['sentiment'].map(sentiment_map)
data

,text,sentiment
0,"[35, 2, 49, 6, 401]",1
1,"[36, 37, 28, 211, 212]",1
2,"[7, 402]",0
3,"[403, 213, 91, 404, 405, 406, 407]",0
4,"[16, 5, 92, 5]",0
...,...,...
494,"[1272, 1273, 1274, 1275, 61, 1276]",0
495,"[119, 1277, 1278, 8, 30]",0
496,"[1279, 1280, 1281, 346]",2
497,"[1282, 1283, 1284, 1285, 147, 361]",1


## Padding


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
len(tok.word_index)

1286

In [ ]:
review_lengths = [len(a) for a in data["text"]]

max_len = max(review_lengths)
print("Maximum sequence length:", max_len)

Maximum sequence length: 18


## Train/Test Split

In [ ]:
X = pad_sequences(data["text"],maxlen=18,padding="post")

In [ ]:
y = data.sentiment

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (315, 18)
X_test shape: (79, 18)


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

### RNN

In [ ]:
vocab_size = len(tok.word_index) + 1  # 1287
embedding_dim = 32

# Redefine the model with return_sequences=True and 5 output units
model_RNN = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=18),
    layers.SimpleRNN(128, activation='tanh', return_sequences=True),
    layers.SimpleRNN(256, activation='tanh'),
    layers.Dense(3, activation='softmax')
])

model_RNN.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_RNN.fit(X_train, y_train, epochs=15, validation_split=0.2)

Epoch 1/15


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - accuracy: 0.4008 - loss: 1.1237 - val_accuracy: 0.4603 - val_loss: 1.1733
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7778 - loss: 0.6538 - val_accuracy: 0.3651 - val_loss: 1.8442
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8929 - loss: 0.2785 - val_accuracy: 0.2540 - val_loss: 2.6272
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9762 - loss: 0.1204 - val_accuracy: 0.3016 - val_loss: 2.8523
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9762 - loss: 0.0719 - val_accuracy: 0.3175 - val_loss: 2.8667
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.9921 - loss: 0.0359 - val_accuracy: 0.3810 - val_loss: 3.0562
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.9960 - loss: 0.0136 - val_accuracy: 0.3492 - val_loss: 3.3532
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9960 - loss: 0.0157 - val_accuracy: 0.3810 - val_loss: 3.4054
Epoch 9/15


In [ ]:
model_RNN.evaluate(X_test, y_test)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5190 - loss: 2.6146


[2.614647626876831, 0.5189873576164246]

In [ ]:
model_RNN.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 18, 32)         │        41,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ (None, 18, 128)        │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ (None, 256)            │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 483,371 (1.84 MB)

 Trainable params: 161,123 (629.39 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 322,248 (1.23 MB)

In [ ]:
# --- label mapping ---
classes = {
    0: "Negative",
    1: "Positive",
    2: "Neutral",
}

# --- prediction function ---
def predict_sentiment(text):
    text = text.lower()
    seq = tok.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=18, padding="post")
    pred = model_RNN.predict(padded)
    print("Prediction probs:", pred)   # 👈 dekho output distribution
    label_idx = pred.argmax(axis=1)[0]
    sentiment = classes[label_idx]
    confidence = round(float(pred.max()) * 100, 2)
    return f"Sentiment: {sentiment} ({confidence}% confidence)"

# --- Gradio UI ---
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(label="Enter a sentence"),
    outputs=gr.Textbox(label="Predicted Sentiment"),
    title="Sentiment Analysis (Keras Model)",
    description="Predicts whether a text is Positive, Negative, or Neutral"
)

# Fix: Launch the interface so that it loads inside your notebook
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f62f059529023b7d2a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### LSTM

In [ ]:
vocab_size = len(tok.word_index) + 1  # 1287
embedding_dim = 32

# Redefine the model with return_sequences=True and 5 output units
model_LSTM = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=18),
    layers.LSTM(128, activation='tanh', return_sequences=True),
    layers.LSTM(256, activation='tanh'),
    layers.Dense(3, activation='softmax')
])

model_LSTM.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_LSTM.fit(X_train, y_train, epochs=15, validation_split=0.2)

Epoch 1/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 179ms/step - accuracy: 0.3611 - loss: 1.0954 - val_accuracy: 0.4127 - val_loss: 1.0831
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 229ms/step - accuracy: 0.3968 - loss: 1.0892 - val_accuracy: 0.3968 - val_loss: 1.0783
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step - accuracy: 0.5000 - loss: 1.0361 - val_accuracy: 0.4127 - val_loss: 1.1480
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 105ms/step - accuracy: 0.5873 - loss: 0.8406 - val_accuracy: 0.3968 - val_loss: 1.2136
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.5635 - loss: 0.7528 - val_accuracy: 0.4127 - val_loss: 1.2351
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 169ms/step - accuracy: 0.6706 - loss: 0.5882 - val_accuracy: 0.4127 - val_loss: 1.3618
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 192ms/step - accuracy: 0.6667 - loss: 0.5137 - val_accuracy: 0.3968 - val_loss: 1.6367
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 104ms/step - accuracy: 0.7500 - loss: 0.4280 - val_accuracy: 0.3810 - val_loss:

In [ ]:
model_LSTM.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 18, 32)         │        41,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 18, 128)        │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,555,883 (5.94 MB)

 Trainable params: 518,627 (1.98 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,037,256 (3.96 MB)

In [ ]:
# --- label mapping ---
classes = {
    0: "Negative",
    1: "Positive",
    2: "Neutral",
}

# --- prediction function ---
def predict_sentiment(text):
    text = text.lower()
    seq = tok.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=18, padding="post")
    pred = model_LSTM.predict(padded)
    print("Prediction probs:", pred)   # 👈 dekho output distribution
    label_idx = pred.argmax(axis=1)[0]
    sentiment = classes[label_idx]
    confidence = round(float(pred.max()) * 100, 2)
    return f"Sentiment: {sentiment} ({confidence}% confidence)"

# --- Gradio UI ---
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(label="Enter a sentence"),
    outputs=gr.Textbox(label="Predicted Sentiment"),
    title="Sentiment Analysis (Keras Model)",
    description="Predicts whether a text is Positive, Negative, or Neutral"
)

# Launch the interface
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://acd601be554d02053c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### GRU

In [ ]:
vocab_size = len(tok.word_index) + 1  # 1287
embedding_dim = 32

# Redefine the model with return_sequences=True and 5 output units
model_GRU = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=18),
    layers.GRU(128, activation='tanh', return_sequences=True),
    layers.GRU(256, activation='tanh'),
    layers.Dense(3, activation='softmax')
])

model_GRU.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_GRU.fit(X_train, y_train, epochs=15, validation_split=0.2)

Epoch 1/15


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 9s 162ms/step - accuracy: 0.3849 - loss: 1.0974 - val_accuracy: 0.4127 - val_loss: 1.0819
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 94ms/step - accuracy: 0.3929 - loss: 1.0911 - val_accuracy: 0.4127 - val_loss: 1.0838
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.3929 - loss: 1.0894 - val_accuracy: 0.4127 - val_loss: 1.0840
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.3929 - loss: 1.0876 - val_accuracy: 0.4127 - val_loss: 1.0818
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - accuracy: 0.4802 - loss: 1.0316 - val_accuracy: 0.3651 - val_loss: 1.2170
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.6071 - loss: 0.7340 - val_accuracy: 0.3810 - val_loss: 1.2419
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.6667 - loss: 0.5954 - val_accuracy: 0.4603 - val_loss: 1.8965
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.6706 - loss: 0.4954 - val_accuracy: 0.4127 - val_loss: 2.0196
Epoch 9/15

In [ ]:
model_GRU.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 18, 32)         │        41,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 18, 128)        │        62,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 256)            │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,201,835 (4.58 MB)

 Trainable params: 400,611 (1.53 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 801,224 (3.06 MB)

In [ ]:
# --- label mapping ---
classes = {
    0: "Negative",
    1: "Positive",
    2: "Neutral",
}

# --- prediction function ---
def predict_sentiment(text):
    text = text.lower()
    seq = tok.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=18, padding="post")
    pred = model_GRU.predict(padded)
    print("Prediction probs:", pred)   # 👈 dekho output distribution
    label_idx = pred.argmax(axis=1)[0]
    sentiment = classes[label_idx]
    confidence = round(float(pred.max()) * 100, 2)
    return f"Sentiment: {sentiment} ({confidence}% confidence)"

# --- Gradio UI ---
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(label="Enter a sentence"),
    outputs=gr.Textbox(label="Predicted Sentiment"),
    title="Sentiment Analysis (Keras Model)",
    description="Predicts whether a text is Positive, Negative, or Neutral"
)

# Launch the interface
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2525e0810a8235cc79.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
